# 01. Розвідковий аналіз даних (EDA) — DVM-CAR

Перед тим як будувати модель, яка за фото автомобіля передбачає рік випуску та ціну, варто спершу подивитись на самі дані "неозброєним оком". У цьому ноутбуці ми не навчаємо жодних моделей — лише досліджуємо маніфест: розподіли цільових змінних (`year`, `price_gbp`), дисбаланс брендів/моделей, ракурси фото, наявність дублікатів та коректність дизайну спліту train/val/test/holdout.

Це важливо з двох причин. По-перше, розуміння розподілів підказує, які трансформації (наприклад, `log(price)`) і які метрики мають сенс. По-друге, EDA — це місце, де ми заздалегідь ловимо потенційні джерела витоку даних (data leakage) та "шорткатів" (shortcuts), якими модель могла б скористатись замість того, щоб навчитись справжніх візуальних ознак.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

DATA_ROOT = "/data/car-price-vision"
MANIFEST = DATA_ROOT + "/manifest.csv"

df = pd.read_csv(MANIFEST)

print("Форма маніфесту (rows, cols):", df.shape)
display(df.head())
df.info()

missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("\nКількість пропущених значень по колонках (лише ненульові):")
print(missing)

## Розподіл року випуску (`year`)

`year` — це рік реєстрації/випуску автомобіля (Reg_year), одна з двох цільових змінних моделі. Зверніть увагу: маніфест уже відфільтрований скриптом `scripts/build_manifest.py` за діапазоном `[1950, 2021]`, тому екстремальних викидів (на кшталт року 1900 чи 2099) тут бути не повинно.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df["year"], bins=50, color="steelblue", edgecolor="black")
ax.set_title("Розподіл року випуску автомобіля (year)")
ax.set_xlabel("Рік випуску (year)")
ax.set_ylabel("Кількість зображень")
plt.tight_layout()
plt.show()

## Ціна (`price_gbp`) та рік оголошення (`advert_year`)

`price_gbp` — друга цільова змінна, ціна оголошення у фунтах стерлінгів. Ціни на авто зазвичай мають правосторонньо скошений (right-skewed) розподіл: багато недорогих авто і невелика кількість дуже дорогих. Тому дивимось і на "сирий" `price_gbp` (з обрізанням верхнього 1% для читабельності гістограми), і на `log_price = log(price_gbp)`, який часто ближчий до нормального і зручніший як ціль регресії.

Окремо варто подивитись на `advert_year` — рік, коли було розміщене оголошення (не плутати з `year`, роком випуску авто). Це важливо, бо ціни в GBP протягом кількох років зазнають впливу інфляції та загальної динаміки ринку вживаних авто: та сама модель, виставлена на продаж у різні роки, може коштувати по-різному навіть без зміни її "справжньої" вартості. Це потенційний confound (супутній фактор), який варто враховувати — наприклад, використовуючи `advert_year` як контрольну змінну, перш ніж робити висновки про зв'язок дизайну авто з ціною.

In [ ]:
price_cutoff = df["price_gbp"].quantile(0.99)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df.loc[df["price_gbp"] <= price_cutoff, "price_gbp"], bins=60, color="darkorange", edgecolor="black")
ax.set_title("Розподіл ціни (price_gbp), обрізано на 99-му перцентилі")
ax.set_xlabel("Ціна, GBP")
ax.set_ylabel("Кількість зображень")
plt.tight_layout()
plt.show()

In [ ]:
df["log_price"] = np.log(df["price_gbp"])

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df["log_price"], bins=60, color="seagreen", edgecolor="black")
ax.set_title("Розподіл log(price_gbp)")
ax.set_xlabel("log(price_gbp)")
ax.set_ylabel("Кількість зображень")
plt.tight_layout()
plt.show()

df[["price_gbp", "log_price"]].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df["advert_year"], bins=df["advert_year"].nunique(), color="slateblue", edgecolor="black")
ax.set_title("Розподіл року оголошення (advert_year)")
ax.set_xlabel("Рік оголошення (advert_year)")
ax.set_ylabel("Кількість зображень")
plt.tight_layout()
plt.show()

## Бренди, моделі та дублікати фото

Дивимось на дисбаланс за брендами (топ-15 за кількістю рядків/зображень), загальну кількість унікальних брендів/моделей/оголошень, а також на те, скільки фото припадає в середньому на одне оголошення (`adv_id`).

Останнє одразу підводить до важливого спостереження: одне оголошення часто містить кілька фото того самого автомобіля з різних ракурсів — фактично майже-дублікати з точки зору ціни й року (усі фото одного оголошення мають однакову ціну і рік випуску). Це критично для дизайну спліту: якщо випадково розкидати окремі *зображення* по train/val/test, кілька фото того самого авто можуть опинитися одночасно і в train, і в test, і модель "запам'ятає" конкретний автомобіль замість того, щоб узагальнювати. Тому спліт нижче групується за `adv_id` (а для суворішої перевірки — і за моделлю авто).

In [ ]:
top_brands = df["brand"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(9, 6))
top_brands.sort_values().plot(kind="barh", ax=ax, color="teal")
ax.set_title("Топ-15 брендів за кількістю зображень")
ax.set_xlabel("Кількість зображень")
ax.set_ylabel("Бренд")
plt.tight_layout()
plt.show()

print("Унікальних брендів:", df["brand"].nunique())
print("Унікальних моделей:", df["model"].nunique())
print("Унікальних оголошень (adv_id):", df["adv_id"].nunique())

In [ ]:
images_per_advert = df.groupby("adv_id").size()
print("Опис розподілу кількості зображень на одне оголошення:")
print(images_per_advert.describe())

dup_adverts = images_per_advert[images_per_advert > 1]
print(f"\nОголошень з більш ніж 1 фото: {len(dup_adverts)} з {len(images_per_advert)} "
      f"({len(dup_adverts) / len(images_per_advert):.1%})")

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(images_per_advert, bins=range(1, int(images_per_advert.max()) + 2), color="firebrick", edgecolor="black")
ax.set_title("Кількість зображень на одне оголошення (adv_id)")
ax.set_xlabel("Кількість зображень в оголошенні")
ax.set_ylabel("Кількість оголошень")
plt.tight_layout()
plt.show()

## Ціна проти року випуску

Очікувано, новіші авто в середньому дорожчі. Перевіримо це на діаграмі розсіювання `log_price` проти `year` (вибірка до 20 000 точок для швидкості відображення).

In [ ]:
sample_df = df if len(df) <= 20_000 else df.sample(n=20_000, random_state=42)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(sample_df["year"], sample_df["log_price"], s=4, alpha=0.2, color="navy")
ax.set_title("log(price_gbp) проти року випуску (year)")
ax.set_xlabel("Рік випуску (year)")
ax.set_ylabel("log(price_gbp)")
plt.tight_layout()
plt.show()

## Ціна в межах бренду (within-brand)

Тепер подивимось на розподіл `log_price` окремо для кількох найпопулярніших брендів. Це важлива перевірка перед моделюванням: якщо ціни сильно відрізняються *між* брендами, модель, яка отримує лише фото, теоретично могла б навчитись впізнавати бренд за візуальними ознаками (шильдик, форма кузова, характерний дизайн) і використовувати його як "шорткат" для передбачення ціни — замість того, щоб вивчати більш тонкі, узагальнювані сигнали дизайну. Це саме той ризик "brand shortcut", про який згадується в README, і саме тому дизайн спліту з holdout невідомих моделей (нижче) важливий для чесної оцінки моделі.

In [ ]:
top5_brands = df["brand"].value_counts().head(5).index.tolist()
data_by_brand = [df.loc[df["brand"] == b, "log_price"].dropna().values for b in top5_brands]

fig, ax = plt.subplots(figsize=(9, 6))
ax.boxplot(data_by_brand, tick_labels=top5_brands, showfliers=False)
ax.set_title("log(price_gbp) для топ-5 брендів")
ax.set_xlabel("Бренд")
ax.set_ylabel("log(price_gbp)")
plt.tight_layout()
plt.show()

## Ракурси зображень (`viewpoint`)

`viewpoint` — передбачений ракурс зображення (-1 означає, що ракурс невідомий, зображення не було злите з `Image_table.csv`). `is_front` показує, чи фото походить з підмножини підтверджених "фронтальних" знімків (`confirmed_fronts`). Ці поля важливі для оцінки того, наскільки однорідні/різноманітні ракурси в датасеті — це впливає на те, чи модель бачить авто з порівнюваних кутів.

In [ ]:
viewpoint_counts = df["viewpoint"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 5))
viewpoint_counts.plot(kind="bar", ax=ax, color="slategray")
ax.set_title("Розподіл ракурсів (viewpoint), -1 = невідомо")
ax.set_xlabel("Код ракурсу (viewpoint)")
ax.set_ylabel("Кількість зображень")
plt.tight_layout()
plt.show()

front_share = df["is_front"].mean()
print(f"Частка зображень з підтвердженого фронтального набору (is_front=True): {front_share:.2%}")

## Спліт train/val/test/holdout без витоку даних

Перевіримо режим `by_model` з `car_price_vision.data.splits.make_splits`: частина моделей авто (`genmodel_id`) повністю виноситься в `holdout` і ніколи не потрапляє в train/val/test — це строгий тест на узагальнення на нові, невідомі моделі. Решта рядків розбивається на train/val/test групуванням за `adv_id`, так що жодне оголошення не "розрізається" між сплітами.

Нижче ми не просто викликаємо функцію, а явно перевіряємо (`assert`) обидві гарантії відсутності витоку: (1) немає перетину `adv_id` між train/val/test, (2) немає перетину `genmodel_id` між holdout і рештою даних.

In [ ]:
sys.path.insert(0, os.path.join("..", "src"))

from car_price_vision.data.splits import make_splits

splits = make_splits(df, mode="by_model")

for name in ("train", "val", "test", "holdout"):
    print(f"{name}: {len(splits[name])} зображень")

adv_train = set(df.iloc[splits["train"]]["adv_id"])
adv_val = set(df.iloc[splits["val"]]["adv_id"])
adv_test = set(df.iloc[splits["test"]]["adv_id"])

assert adv_train.isdisjoint(adv_val), "Витік: спільні adv_id між train і val"
assert adv_train.isdisjoint(adv_test), "Витік: спільні adv_id між train і test"
assert adv_val.isdisjoint(adv_test), "Витік: спільні adv_id між val і test"

model_holdout = set(df.iloc[splits["holdout"]]["genmodel_id"])
model_rest = set(df.iloc[np.concatenate([splits["train"], splits["val"], splits["test"]])]["genmodel_id"])

assert model_holdout.isdisjoint(model_rest), "Витік: спільні genmodel_id між holdout і рештою даних"

print("Усі перевірки на відсутність витоку даних пройдені успішно.")